In [9]:
import pandas as pd
import numpy as np
import os

os.getcwd()

'c:\\Users\\Charlie\\6013\\notebooks'

In [10]:
summaries = {
    "mtl_original": pd.read_json("../outputs/eval_summary_mtl_mtl_original.json"),
    "mtl_boosted": pd.read_json("../outputs/eval_summary_mtl_mtl_boosted.json"),
    "stl_aspect": pd.read_json("../outputs/eval_summary_stl_aspect_original.json"),
    "stl_aspect_sentiment": pd.read_json("../outputs/eval_summary_stl_aspect_sentiment_original.json"),
    "stl_bug_report": pd.read_json("../outputs/eval_summary_stl_bug_report_original.json"),
    "stl_feature_request": pd.read_json("../outputs/eval_summary_stl_feature_request_original.json"),
}
for task, df in summaries.items():
    print(df.head())




                 mode   dataset task                          model_path  \
bug_report        mtl  original  all  outputs/best_model_mtl_original.pt   
feature_request   mtl  original  all  outputs/best_model_mtl_original.pt   
aspect            mtl  original  all  outputs/best_model_mtl_original.pt   
aspect_sentiment  mtl  original  all  outputs/best_model_mtl_original.pt   

                                                            results  
bug_report        {'macro_f1': 0.7833333333333331, 'macro_precis...  
feature_request   {'macro_f1': 0.7632819746470161, 'macro_precis...  
aspect            {'macro_f1': 0.7170467094024511, 'macro_precis...  
aspect_sentiment  {'macro_f1': 0.7574652640875611, 'macro_precis...  
                 mode  dataset task                         model_path  \
bug_report        mtl  boosted  all  outputs/best_model_mtl_boosted.pt   
feature_request   mtl  boosted  all  outputs/best_model_mtl_boosted.pt   
aspect            mtl  boosted  all  outputs/be

In [11]:
print(type(summaries["mtl_original"]))
print(summaries["mtl_original"].columns)
print(summaries["mtl_original"].iloc[0])

<class 'pandas.core.frame.DataFrame'>
Index(['mode', 'dataset', 'task', 'model_path', 'results'], dtype='object')
mode                                                        mtl
dataset                                                original
task                                                        all
model_path                   outputs/best_model_mtl_original.pt
results       {'macro_f1': 0.7833333333333331, 'macro_precis...
Name: bug_report, dtype: object


In [12]:
# Creating section 1, general summary table of macro_f1, macro_precision, macro_recall, accuracy inside per_class with confidence values
"""
{
    "mode": "mtl",
    "dataset": "original",
    "task": "all",
    "model_path": "outputs/best_model_mtl_original.pt",
    "results": {
        "bug_report": {
            "macro_f1": 0.7833333333333333,
            "macro_precision": 0.7684555303602922,
            "macro_recall": 0.8027848820687695,
            "confidence": {
                "overall": 0.9568860530853271,
                "correct": 0.97133868932724,
                "incorrect": 0.8671128153800964
            },
            "per_class": {
                "No": {
                    "precision": 0.9319727891156463,
                    "recall": 0.8954248366013072,
                    "f1-score": 0.9133333333333333,
                    "support": 612.0
                },
                "Yes": {
                    "precision": 0.6049382716049383,
                    "recall": 0.7101449275362319,
                    "f1-score": 0.6533333333333333,
                    "support": 138.0
                },
                "accuracy": 0.8613333333333333,
                "macro avg": {
                    "precision": 0.7684555303602922,
                    "recall": 0.8027848820687695,
                    "f1-score": 0.7833333333333333,
                    "support": 750.0
                },
                "weighted avg": {
                    "precision": 0.8717984378936761,
                    "recall": 0.8613333333333333,
                    "f1-score": 0.8654933333333333,
                    "support": 750.0
                }
            }
        },
"""



rows = []
# "mtl_original", pd.read_json("../outputs/eval_summary_mtl_mtl_original.json")
for model_name, df in summaries.items():
    
    for task_name, row in df.iterrows():
    
        task_result = row["results"]

        rows.append({
            "model": model_name,
            "task": task_name,

            "macro_f1": task_result["macro_f1"],
            "macro_precision": task_result["macro_precision"],
            "macro_recall": task_result["macro_recall"],
            "accuracy": task_result["per_class"]["accuracy"],

            "conf_overall": task_result["confidence"]["overall"],
            "conf_correct": task_result["confidence"]["correct"],
            "conf_incorrect": task_result["confidence"]["incorrect"],

        })

analysis_summary_df = pd.DataFrame(rows)
print(analysis_summary_df.head())
analysis_summary_df.to_csv("analysis.csv", index=False)

          model              task  macro_f1  macro_precision  macro_recall  \
0  mtl_original        bug_report  0.783333         0.768456      0.802785   
1  mtl_original   feature_request  0.763282         0.752334      0.776279   
2  mtl_original            aspect  0.717047         0.718659      0.717574   
3  mtl_original  aspect_sentiment  0.757465         0.771312      0.747191   
4   mtl_boosted        bug_report  0.905186         0.901647      0.909222   

   accuracy  conf_overall  conf_correct  conf_incorrect  
0  0.861333      0.956886      0.971339        0.867113  
1  0.869333      0.960423      0.971002        0.890037  
2  0.736000      0.898030      0.927060        0.817099  
3  0.912000      0.965079      0.976294        0.848845  
4  0.913218      0.975117      0.983552        0.886357  


In [13]:
pivot = analysis_summary_df.pivot(
    index="task",
    columns="model",
    values="macro_f1"
)

print(pivot) # wrong, pull actual stl score

pivot["stl"] = None

pivot.loc["bug_report", "stl"] = pivot.loc["bug_report", "stl_bug_report"]
pivot.loc["feature_request", "stl"] = pivot.loc["feature_request", "stl_feature_request"]
pivot.loc["aspect", "stl"] = pivot.loc["aspect", "stl_aspect"]
pivot.loc["aspect_sentiment", "stl"] = pivot.loc["aspect_sentiment", "stl_aspect_sentiment"]

pivot["mtl_orig_excluding_stl"] = pivot["mtl_original"] - pivot["stl"]
pivot["mtl_boost_excluding_orig"] = pivot["mtl_boosted"] - pivot["mtl_original"]
final = pivot[
    [
        "stl",
        "mtl_original",
        "mtl_boosted",
        "mtl_orig_excluding_stl",
        "mtl_boost_excluding_orig"
    ]
]

print("\nfinal:\n\n", final)

final.to_csv("analysis_macro_f1_pivot.csv")



model             mtl_boosted  mtl_original  stl_aspect  stl_aspect_sentiment  \
task                                                                            
aspect               0.802578      0.717047    0.694127                   NaN   
aspect_sentiment     0.600339      0.757465         NaN              0.785615   
bug_report           0.905186      0.783333         NaN                   NaN   
feature_request      0.816422      0.763282         NaN                   NaN   

model             stl_bug_report  stl_feature_request  
task                                                   
aspect                       NaN                  NaN  
aspect_sentiment             NaN                  NaN  
bug_report              0.784503                  NaN  
feature_request              NaN             0.741924  

final:

 model                  stl  mtl_original  mtl_boosted mtl_orig_excluding_stl  \
task                                                                           
aspect 

In [14]:
per_class_rows = []
skip = {"accuracy", "macro avg", "weighted avg"}

for model_name, df in summaries.items():
    for task_name, row in df.iterrows():
        task_result = row["results"]
        
        for class_name, class_metrics in task_result["per_class"].items():
            if class_name in skip:
                continue
            per_class_rows.append({
                "model":     model_name,
                "task":      task_name,
                "class":     class_name,
                "precision": class_metrics["precision"],
                "recall":    class_metrics["recall"],
                "f1":        class_metrics["f1-score"],
                "support":   class_metrics["support"],
            })

per_class_analysis_df = pd.DataFrame(per_class_rows)
print("Per class df", per_class_analysis_df)
per_class_analysis_df.to_csv("per_class_analysis.csv", index=False)


Per class df                    model              task     class  precision    recall  \
0           mtl_original        bug_report        No   0.931973  0.895425   
1           mtl_original        bug_report       Yes   0.604938  0.710145   
2           mtl_original   feature_request        No   0.932149  0.911532   
3           mtl_original   feature_request       Yes   0.572519  0.641026   
4           mtl_original            aspect       App   0.787500  0.840000   
5           mtl_original            aspect    Driver   0.786260  0.780303   
6           mtl_original            aspect   General   0.803738  0.688000   
7           mtl_original            aspect   Payment   0.617647  0.636364   
8           mtl_original            aspect   Pricing   0.698630  0.750000   
9           mtl_original            aspect   Service   0.618182  0.610778   
10          mtl_original  aspect_sentiment  Positive   0.938776  0.953368   
11          mtl_original  aspect_sentiment   Neutral   0.451613

In [15]:
label_names = {
    'bug_report': ['No', 'Yes'],
    'feature_request': ['No', 'Yes'],
    'aspect': ['App', 'Driver', 'General', 'Payment', 'Pricing', 'Service'],
    'aspect_sentiment': ['Positive', 'Neutral', 'Negative']
}

In [16]:
tasks = ["bug_report", "feature_request", "aspect", "aspect_sentiment"]
mcnemar_rows = []

mtl_path = "../outputs/test_predictions_mtl_mtl_original.csv"
mtl_df = pd.read_csv(mtl_path)

from statsmodels.stats.contingency_tables import mcnemar

In [17]:
for task in tasks:
    stl_path = f"../outputs/test_predictions_stl_{task}_original.csv"
    
    if not os.path.exists(stl_path):
        print(f"Warning: Missing STL file for {task}")
        continue
        
    stl_df = pd.read_csv(stl_path)

    # 3. Align ground truth
    # The 'task' column in the CSV is numeric (0, 1, 2...). 
    # The '{task}_pred' column is a string ('Yes', 'No', 'App'...).
    # We map the numeric ground truth to labels so we can compare directly.
    y_true = [label_names[task][int(val)] for val in mtl_df[task]]
    
    # 4. Get predictions
    y_mtl = mtl_df[f"{task}_pred"].values
    y_stl = stl_df[f"{task}_pred"].values

    # 5. Determine correctness for each model
    mtl_correct = (y_mtl == y_true)
    stl_correct = (y_stl == y_true)

    # 6. Build the 2x2 Contingency Table
    # Table structure for McNemar:
    #              MTL Correct | MTL Wrong
    # STL Correct |    [0,0]   |   [0,1]
    # STL Wrong   |    [1,0]   |   [1,1]
    
    both_correct = np.sum(mtl_correct & stl_correct)
    stl_only     = np.sum(~mtl_correct & stl_correct) # STL right, MTL wrong
    mtl_only     = np.sum(mtl_correct & ~stl_correct) # MTL right, STL wrong
    both_wrong   = np.sum(~mtl_correct & ~stl_correct)

    contingency_table = [
        [both_correct, stl_only],
        [mtl_only, both_wrong]
    ]

    # 7. Run McNemar's Test
    # We use exact=True because some of our discordant cells (mtl_only/stl_only) might be small
    result = mcnemar(contingency_table, exact=True)
    
    mcnemar_rows.append({
        "task": task,
        "both_correct": both_correct,
        "mtl_only_correct": mtl_only,
        "stl_only_correct": stl_only,
        "both_wrong": both_wrong,
        "p_value": result.pvalue,
        "significant": result.pvalue < 0.05
    })

# 8. Save and Display Results
mcnemar_df = pd.DataFrame(mcnemar_rows)
print("\nMcNemar Test Results (MTL vs STL Original):")
print(mcnemar_df[["task", "mtl_only_correct", "stl_only_correct", "p_value", "significant"]])

mcnemar_df.to_csv("analysis_mcnemar_results.csv", index=False)


McNemar Test Results (MTL vs STL Original):
               task  mtl_only_correct  stl_only_correct   p_value  significant
0        bug_report                32                28  0.698883        False
1   feature_request                42                37  0.652964        False
2            aspect                70                55  0.210327        False
3  aspect_sentiment                15                18  0.728332        False
